In [1]:
import numpy as np
import matplotlib as plt
import seaborn as sea
import pandas as pd
import xgboost as xgb
import sklearn as sk
import sqlite3 
import mysql.connector
import duckdb # for me to use sql even though it is not necessary with the way the datasets are given

## Data Definition

I am asked by Northern Lights Air (NLA) to run an analysis on their loyalty program. NLA ran a promotional campaign from Februrary 2018 (2018-02-01) until April 2018 (2018-04-31). NLA wants to improve their loyalty program, hence they want to know:
- What impact did the campaign have on the loyalty memberships? (Enrollment, Cancellation, Points redeemed etc.)
- What impact did the campaign have on booked flights? (in the summer, in general etc.)
- What impact did it have on customer churning?
- Was the campaign more successful towards certain demographics? (Gender, Education, Marital Status etc.)

## Exploratory Analysis

I will use SQL (for fun) to explore the data. 

**392936** total data entries index starting at 0

### Calendar 
This is a dataset that gives me everyday from 2012-01-01 (January 1, 2012) until 2018-12-31 (December 31, 2018). It gives me:
- Start of Year: first date of the year
- Start of Quarter: first date of the quarter it's in (Jan 1, Apr 1, Jul 1, Oct 1)
- Start of Month: first date of the month it's in

### Customer Flights Activity 
- Loyalty Number: **16737** unique members based on loyalty numbers
- Year
- Month
- Total Flights: Do not have to be loyalty member to fly
- Distance
- Points Accumulated
- Points Redeemed
- Dollar Cost Points Redeemed (CAD)

### Loyalty Program 
- Loyalty Number: **16737** unique members based on loyalty numbers
- Country: Canada (constant)
- Province
- City
- Postal Code
- Gender
- Education
- Salary (CAD): Contains NULL
- Marital Status
- Loyalty Card
- CLV
- Enrollment Type
- Enrollment Year
- Enrollment Month
- Cancellation Year: Contains NULL
- Cancellation Month: Contains NULL

### New Column Ideas
- Enrollment_length: Amount of time you've been in the program

I can see that we can **combine** Enrollment Year and Month together into one column, as well as Cancellation Year and Month. \
Can **combine** the two tables via Loyalty Number. \
Make two different tables, one that matches enrolment date to the calendar and one that matches cancellation date to calendar.\
Notice that column names are strings let's change that


In [33]:
#Quick check to see the tables
calendar_df = pd.read_csv("Calendar.csv")
flight_df = pd.read_csv("Customer Flight Activity.csv")
loyalty_df = pd.read_csv("Customer Loyalty History.csv")

print(flight_df.head())
print(loyalty_df.head())

   Loyalty Number  Year  Month  Total Flights  Distance  Points Accumulated  \
0          100590  2018      6             12     15276             22914.0   
1          100590  2018      7             12      9168             13752.0   
2          100590  2018      5              4      6504              9756.0   
3          100590  2018     10              0         0                 0.0   
4          100590  2018      2              0         0                 0.0   

   Points Redeemed  Dollar Cost Points Redeemed  
0                0                            0  
1                0                            0  
2                0                            0  
3              512                           92  
4                0                            0  
   Loyalty Number Country          Province       City Postal Code  Gender  \
0          480934  Canada           Ontario    Toronto     M2Z 4K1  Female   
1          549612  Canada           Alberta   Edmonton     T3G 6Y6   

In [30]:
#to avoid repetition we can aggregate the stats of each distinct loyalty number via summation
#assign it to dataframe called "flight"
flight = duckdb.sql("""
SELECT 
    "Loyalty Number",
    SUM("Total Flights") AS "Total Flights",
    SUM(Distance) AS "Total Distance",
    SUM("Points Accumulated") AS "Total Points",
    SUM("Points Redeemed") AS "Total Redeemed Points",
    SUM("Dollar Cost Points Redeemed") AS "Total Dollar Points Redeemed"
FROM flight_df 
GROUP BY "Loyalty Number" 
ORDER BY "Loyalty Number"
""").df()

loyalty

### Due Diligence Data Clean and Check

- Null values
- Correct data types
- Correct formatting for datetime

In [32]:
#Join tables 
#Concat Enrollment and Cancellation Month and Year together
#We use day as 01 
#Cancelled is the target (Y)
duckdb.sql(""" 
    SELECT * EXCLUDE("Cancellation Year", "Cancellation Month", Country),
        MAKE_DATE("Enrollment Year", "Enrollment Month", 1) AS "Enrollment Date",
        MAKE_DATE("Cancellation Year", "Cancellation Month", 1) AS "Cancellation Date",
        CASE WHEN "Cancellation Year" IS NULL THEN 1 ELSE 0 END AS Cancelled,
        CASE WHEN "Salary" IS NULL THEN 1 ELSE 0 END AS Salary_Missing
    FROM read_csv_auto('Customer Loyalty History.csv') clh 
    LEFT JOIN flight fl
        on clh."Loyalty Number" = fl."Loyalty Number"
    ORDER BY clh."Loyalty Number"
           """).df()


,Loyalty Number,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,...,Loyalty Number_1,Total Flights,Total Distance,Total Points,Total Redeemed Points,Total Dollar Points Redeemed,Enrollment Date,Cancellation Date,Cancelled,Salary_Missing
0,100018,Alberta,Edmonton,T9G 1W3,Female,Bachelor,92552,Married,Aurora,7919.20,...,100018,46.0,81190.0,81190.0,1513.0,272.0,2016-08-01,NaT,1,0
1,100102,Ontario,Toronto,M1R 4K3,Male,College,<NA>,Single,Nova,2887.74,...,100102,51.0,68918.0,68918.0,1195.0,215.0,2013-03-01,NaT,1,1
2,100140,British Columbia,Dawson Creek,U5I 4F1,Female,College,<NA>,Divorced,Nova,2838.07,...,100140,47.0,72856.0,72856.0,593.0,107.0,2016-07-01,NaT,1,1
3,100214,British Columbia,Vancouver,V5R 1W3,Male,Bachelor,63253,Married,Star,4170.57,...,100214,22.0,38236.0,38236.0,861.0,155.0,2015-08-01,NaT,1,0
4,100272,Ontario,Toronto,P1L 8X8,Female,Bachelor,91163,Divorced,Star,6622.05,...,100272,37.0,54997.0,54997.0,1007.0,182.0,2014-01-01,NaT,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16732,999902,Ontario,Toronto,M1R 4K3,Male,College,<NA>,Married,Aurora,7290.07,...,999902,50.0,83725.0,83725.0,876.0,158.0,2014-05-01,NaT,1,1
16733,999911,Newfoundland,St. John's,A1C 6H9,Male,Doctor,217943,Single,Nova,8564.77,...,999911,0.0,0.0,0.0,0.0,0.0,2012-08-01,NaT,1,0
16734,999940,Quebec,Quebec City,G1B 3L5,Female,Bachelor,47670,Married,Nova,20266.50,...,999940,18.0,28275.0,28275.0,672.0,121.0,2017-07-01,NaT,1,0
16735,999982,British Columbia,Victoria,V10 6T5,Male,College,<NA>,Married,Star,2631.56,...,999982,6.0,8323.0,8323.0,0.0,0.0,2018-07-01,NaT,1,1


In [ ]:
# #double check that you don't have to be a loyalty member to fly
# duckdb.sql(""" 
#     SELECT
#         cfa."Loyalty Number",
#         Year,
#         "Cancellation Year"
#     FROM read_csv_auto('Customer Flight Activity.csv') cfa
#     LEFT JOIN read_csv_auto('Customer Loyalty History.csv') clh
#         on cfa."Loyalty Number" = clh."Loyalty Number"
#     WHERE "Cancellation Year" < "Year"
#            """).df()

In [71]:
#Joining the tables 

duckdb.sql("""
    SELECT *,
        MAKE_DATE(Year, Month, 1) AS "Flight Date",
        MAKE_DATE("Enrollment Year", "Enrollment Month", 1) AS "Enrollment Date",
        MAKE_DATE("Cancellation Year", "Cancellation Month", 1) AS "Cancellation Date"
    FROM read_csv_auto('Customer Flight Activity.csv') cfa
    LEFT JOIN read_csv_auto('Customer Loyalty History.csv') clh
        on cfa."Loyalty Number" = clh."Loyalty Number"
           """).df()


,Loyalty Number,Year,Month,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed,Loyalty Number_1,Country,...,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month,Flight Date,Enrollment Date,Cancellation Date
0,427140,2017,11,0,0,0.0,0,0,427140,Canada,...,Star,2574.02,Standard,2015,1,<NA>,<NA>,2017-11-01,2015-01-01,NaT
1,427177,2017,11,0,0,0.0,0,0,427177,Canada,...,Star,41787.90,Standard,2016,4,<NA>,<NA>,2017-11-01,2016-04-01,NaT
2,427354,2017,11,0,0,0.0,0,0,427354,Canada,...,Star,2429.28,Standard,2017,12,<NA>,<NA>,2017-11-01,2017-12-01,NaT
3,486909,2017,10,0,0,0.0,0,0,486909,Canada,...,Aurora,13864.38,Standard,2014,6,2018,9,2017-10-01,2014-06-01,2018-09-01
4,486909,2017,11,0,0,0.0,0,0,486909,Canada,...,Aurora,13864.38,Standard,2014,6,2018,9,2017-11-01,2014-06-01,2018-09-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
392931,426530,2017,11,0,0,0.0,0,0,426530,Canada,...,Star,2450.63,Standard,2013,3,<NA>,<NA>,2017-11-01,2013-03-01,NaT
392932,426720,2017,11,0,0,0.0,0,0,426720,Canada,...,Aurora,7891.58,Standard,2016,8,<NA>,<NA>,2017-11-01,2016-08-01,NaT
392933,426981,2017,11,0,0,0.0,0,0,426981,Canada,...,Aurora,8573.46,Standard,2015,3,<NA>,<NA>,2017-11-01,2015-03-01,NaT
392934,958005,2017,1,0,0,0.0,491,88,958005,Canada,...,Aurora,7704.25,Standard,2012,11,<NA>,<NA>,2017-01-01,2012-11-01,NaT


In [66]:
# Check for null valuesabs
duckdb.sql("""
SELECT 
    column_name,
    column_type,
    null_percentage
FROM(SUMMARIZE(
        SELECT *
        FROM read_csv_auto('Customer Flight Activity.csv') cfa
        LEFT JOIN read_csv_auto('Customer Loyalty History.csv') clh
            on cfa."Loyalty Number" = clh."Loyalty Number"))
""").df()

#87% of people have not churned away from the loyalty programabs
#25% null values 

,column_name,column_type,null_percentage
0,Loyalty Number,BIGINT,0.00
1,Year,BIGINT,0.00
2,Month,BIGINT,0.00
3,Total Flights,BIGINT,0.00
4,Distance,BIGINT,0.00
5,Points Accumulated,DOUBLE,0.00
6,Points Redeemed,BIGINT,0.00
7,Dollar Cost Points Redeemed,BIGINT,0.00
8,Loyalty Number,BIGINT,0.00
9,Country,VARCHAR,0.00
